# Bounded integer optimization

Minimize `3*x + 2*y` subject to `3*x + y >= 2`, with integers `0 <= x,y <= 3`. We keep one QAOA layer and penalty 20. This native encoding uses the positive-part-squared violation, rather than reproducing a vendor slack circuit. Angles are radians.


In [ ]:
from typing import Annotated

import flagquantum as fq
from examples.synthesis.tutorials.basics.optimization.learning_optimization.overview.workflow import (
    run_workflow,
)
from flagquantum import synth
from flagquantum.synth import (
    RX,
    Output,
    QNum,
    allocate,
    control,
    hadamard_transform,
    phase,
    qfunc,
)


@qfunc
def objective_layer(gamma: float, x: QNum, y: QNum) -> None:
    """Apply the objective and the positive-part-squared inequality penalty."""
    phase(3 * x + 2 * y, -gamma)
    control(
        3 * x + y < 2,
        lambda: phase(20 * (2 - 3 * x - y) * (2 - 3 * x - y), -gamma),
    )


@qfunc
def main(
    gamma: float,
    beta: float,
    x: Output[Annotated[QNum, 2]],
    y: Output[Annotated[QNum, 2]],
) -> None:
    """Prepare uniform integers, apply the cost, and mix each bit."""
    allocate(x)
    allocate(y)
    hadamard_transform(x)
    hadamard_transform(y)
    objective_layer(gamma, x, y)
    for bit in range(2):
        RX(2 * beta, x[bit])
        RX(2 * beta, y[bit])


In [ ]:
program = synth.synthesize(
    main,
    classical_inputs={"gamma": 0.19, "beta": 0.37},
    max_qubits=4,
    max_workspace=0,
    max_operations=1000,
    memory_limit_bytes=1024**2,
)
result = fq.run(program)
print(dict(program.probabilities(result, registers=("x", "y"))))
print(program.report["resources"])


## Optimize actual quantum executions

The helper below reuses `NelderMeadOptimizer`. Every evaluation synthesizes and runs the native program above and reads named probabilities. Its bounded 32 iterations need not converge. An exact classical solution is used only in acceptance tests, never substituted into the quantum search.


In [ ]:
workflow = run_workflow()
print("initial expectation", workflow.trace[0].expectation)
print("final expectation", workflow.optimizer.value)
print("iterations", workflow.optimizer.iterations)
print("evaluations", workflow.optimizer.evaluations)
print("converged", workflow.optimizer.converged)


## Inspect local samples

Draw 1000 local samples from the final executed distribution, with CPU seed 2718. Sort feasible observations by unpenalized cost. The best observed assignment is not a guarantee of variational convergence or a hardware result.


In [ ]:
print("best observed feasible", workflow.best_feasible)
print("feasible sample count", sum(row.count for row in workflow.samples if row.feasible))
for row in workflow.samples:
    print(row)
print(workflow.program.report["resources"])
